# Citi Velocity USTs — EOD

Citi Velocity as the truth source for US Treasuries, through the usual
**MDP → TimeseriesBuilder → Query** path.

Two things make this source different from the other UST sources in the repo:

- **It is a quote service.** Citi publishes eight values per bond and no pricer is
  entitled, so anything Citi does not publish is rebuilt locally in rateslib /
  QuantLib from the quote it does publish. Which of the two you got is recorded on
  every number — see the provenance section below.
- **It carries a liquid subset, not every UST.** 349 nominal coupon bonds,
  0.02y to 29.8y. No bills, no TIPS, no FRNs, no STRIPS. And it lags new auctions
  — see the caveat at the bottom, which is the thing most likely to surprise you.

The whole universe is warmed nightly by
`scripts/citivelo_ust_universe_warm.py`, so everything here reads from cache and
opens no workbook.

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

import pandas as pd
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [ ]:
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue

# "-RL" builds rateslib pricers, "-QL" QuantLib. Both read the same Citi quotes.
usts_mdp = FixedRateBondsMDP(source="USTS_CITIVELO-RL")
ts_builder = TimeseriesBuilder()

## The universe

`BondUniverse` reads the committed catalog — no Excel, no network. Coverage is
**per bond**: every bond has `PRICE`/`YIELD`/`DURATION`/`SPREAD_TSY`, but only
305 of 349 serve `DV01` and 253 serve `ASW_4_USD`. Ask
`available_values(isin)` rather than assuming.

In [ ]:
from MDP.CitiVelocityExcel.bonds import BondUniverse

uni = BondUniverse.from_catalog(country="USA", asset_type="GOVT")
print(f"{len(uni)} bonds, {uni.to_frame()['maturity'].min()} .. {uni.to_frame()['maturity'].max()}")

frame = uni.to_frame()
display(frame.head())

# per-value coverage across the universe
import collections
cov = collections.Counter(v for d in uni for v in uni.available_values(d.isin))
pd.Series(cov).sort_values(ascending=False).to_frame("bonds serving")

## Outright values

`UnifiedQuery(cusip=...)` takes an on-the-run alias (`CT10`, `O5`, `OO2`) or a
literal CUSIP. The alias is resolved against the UST reference table first, then
the CUSIP is converted to Citi's ISIN — `US` + CUSIP + ISO 6166 check digit,
verified against all 2,162 ISINs Citi serves.

In [ ]:
start = datetime.date(2026, 7, 20)
end   = datetime.date(2026, 8, 7)

queries = [
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_YTM),
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_CLEAN_PRICE),
    UnifiedQuery(cusip="CT10", value=UnifiedValue.FRB_SPREAD_TSY),
]

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=queries,
    routers={
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    ignore_cache_miss=True,
)
df

In [ ]:
plot, fig, _, _, legend = make_secondary_axis_plot(engine="plotly")
plot(df[queries[0].col_name()], which="left")
plot(df[queries[2].col_name()], which="right")
legend(valfmt="{:.4f}", show_date=True)

## Quoted or computed? — provenance

`FRB_SPREAD_TSY`, `FRB_OAS`, `FRB_ASW_SPREAD` and `FRB_CAS` are **quote-only**:
they return Citi's published number and raise on a pricer from any other source,
rather than returning `0.0`, which is a perfectly plausible spread.

Everything else is **computed locally** from Citi's `PRICE`. The pricer records
which, so a surprising number is traceable instead of needing to be re-derived.

The four readings that could have been confidently wrong were settled by
measurement, not by the tag name — by asking which reading of one Citi number
reproduces another:

| value | verdict | margin |
|---|---|---|
| `PRICE` | **clean**, per 100 | 0.0186 bp vs Citi's own YIELD; **57.43 bp** if read as dirty |
| `DURATION` | **modified** | 1.9e-05 yr; **0.0500 yr** if read as Macaulay |
| `DV01` | **per 1mm face**, + for a long | ratio to local per-100 dv01 = 10000.06 |

In [ ]:
from MDP.CitiVelocityExcel.bonds import values as V

pricers = usts_mdp.get_data({"cusips": ["CT10"], "timestamp": end})
meta = pricers["CT10"].meta()

rows = []
for name in ("CLEAN_PRICE", "YTM", "MOD_DURATION", "DV01", "SPREAD_TSY"):
    p = V.provenance_of(meta, name)
    if p is not None:
        rows.append({"value": name, "origin": p.origin, "unit": p.unit,
                     "verified": p.verified, "detail": p.detail})
display(pd.DataFrame(rows))

# What Citi published for this bond, verbatim and unscaled:
pd.Series(V.quoted_value_of(meta, v) for v in ("PRICE", "YIELD", "DURATION", "DV01")), \
    V.coverage_of(meta)

## Curves and flies

Two legs make a `CURVE`, three a `FLY`. The `/` spelling is the same one the
other UST notebooks use.

In [ ]:
curve_q = UnifiedQuery(cusip="CT5/CT10", value=UnifiedValue.FRB_YTM)
fly_q   = UnifiedQuery(cusip="CT2/CT5/CT10", value=UnifiedValue.FRB_YTM)

curves = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=[curve_q, fly_q],
    routers={"FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True)},
    n_jobs=12,
    ignore_cache_miss=True,
)
curves

## Caveat — Citi lags new auctions

Measured 2026-08-08 against the repo's own `fiscaldata` reference table: **352**
live nominal coupon USTs, of which Citi carried **349**. The six absent were
three when-issued (settling 2026-08-17, correctly not quoted) and **three issued
2026-07-31 — eight days earlier — that Citi had still not picked up.**

Those three are the on-the-run 2Y, 5Y and 7Y, so `UnifiedQuery(cusip="CT2")`
resolves to a real bond that this source cannot quote. Resolution raises a
coverage error naming it rather than returning nothing, so it is loud — but it
means **the freshest on-the-run may not be available here**. Use the previous
issue (`O2`) or another source when you need the very newest bond.

`scripts/citivelo_ust_universe_warm.py refresh` picks up new bonds as soon as
Citi has them, and never removes: matured bonds are unrecoverable from
`CVCURVEBOND` (asking it for an old date returns today's set filtered, not the
set as it stood), so anything seen once is kept and its cached history stays
valid.

In [ ]:
from MDP.CitiVelocityExcel.bonds.resolution import resolve_bond

# resolve_bond takes a CUSIP or an ISIN. On-the-run ALIASES (CT10, O2) are
# resolved to a CUSIP first, by FixedRateBondsMDP against the UST reference
# table -- there is one alias table in this repo, not a second one here.
for cusip, what in [
    ("91282CNJ6", "a bond Citi carries"),
    ("91282CRA1", "the 5Y issued 2026-07-31 -- Citi has not picked it up"),
    ("037833100", "Apple: a real ISIN, valid check digit, no rates desk quotes it"),
]:
    try:
        r = resolve_bond(cusip)
        print(f"{cusip:<12} {what:<52} -> {r.isin}  {r.descriptor.description}")
    except Exception as e:
        print(f"{cusip:<12} {what:<52} -> {type(e).__name__}")
        print(f"{'':<12} {str(e)[:150]}")